In [1]:
import json
from timeeval import TimeEval, DefaultMetrics, Algorithm, TrainingType, InputDimensionality, ResourceConstraints
from timeeval.datasets.dataset_manager import DatasetManager, DatasetRecord
from timeeval.adapters import DockerAdapter
from timeeval.params import FixedParameters
from timeeval.resource_constraints import GB
from timeeval.metrics import F1Score, Precision, Recall
from timeeval.metrics.thresholding import FixedValueThresholding, NoThresholding
from timeeval.metrics.thresholding import PercentileThresholding
import json
from pathlib import Path
import pandas as pd

with open("config.json", "r") as f:
    config = json.load(f)

path_to_timeeval_datasets = config["path_to_timeeval_datasets"]
collection_name = config["collection_name"]
dataset_name = config["dataset_name"]

In [2]:
dm = DatasetManager(path_to_timeeval_datasets, create_if_missing=True)
meta = json.load(open(f"{path_to_timeeval_datasets}/{collection_name}/{dataset_name}/{dataset_name}.metadata.json","r"))[0]

record = DatasetRecord(
    collection_name=collection_name,
    dataset_name=dataset_name,
    train_path=None,
    test_path=f"{collection_name}/{dataset_name}/{dataset_name}.test.csv",
    dataset_type="real",
    datetime_index=False,
    split_at=None,
    train_type="unsupervised",
    train_is_normal=False,
    input_type="univariate" if meta["dimensions"] == 1 else "multivariate",
    length=meta["length"],
    dimensions=meta["dimensions"],
    contamination=meta["contamination"],
    num_anomalies=meta["num_anomalies"],
    min_anomaly_length=meta["anomaly_length"]["min"],
    median_anomaly_length=meta["anomaly_length"]["median"],
    max_anomaly_length=meta["anomaly_length"]["max"],
    mean=meta["means"],
    stddev=meta["stddevs"],
    trend=meta["trends"],
    stationarity=meta["stationarities"],
    period_size=0
)

dm.add_dataset(record)

selected_datasets = dm.select(collection=collection_name)

In [3]:
def preprocess(data, args):
    # Other algorithms are supposed to just receive the "value" column
    df = pd.read_csv(data)
    df = df[["timestamp", "nice_battery_mv", "is_anomaly"]]
    output_path = data.parent / f"{data.stem}_preprocessed.csv"
    df.to_csv(output_path, index=False)
    return output_path


algorithms = [
    Algorithm(
    name="pcc",
    main=DockerAdapter(image_name="ghcr.io/timeeval/pcc", tag="0.3.1", skip_pull=True),
    preprocess=preprocess,
    data_as_file=True,
    training_type=TrainingType.UNSUPERVISED,
    input_dimensionality=InputDimensionality.MULTIVARIATE
    ),
    Algorithm(
    name="hbos",
    main=DockerAdapter(image_name="hbos", tag="0.3.1", skip_pull=True),
    preprocess=preprocess,
    param_config=FixedParameters({"n_bins": 50}),
    data_as_file=True,
    training_type=TrainingType.UNSUPERVISED,
    input_dimensionality=InputDimensionality.MULTIVARIATE
    ),
    Algorithm(
    name="iforest",
    main=DockerAdapter(image_name="ghcr.io/timeeval/iforest", tag="0.3.1", skip_pull=True),
    preprocess=preprocess,
    data_as_file=True,
    training_type=TrainingType.UNSUPERVISED,
    input_dimensionality=InputDimensionality.MULTIVARIATE
    ),
    Algorithm(
    name="knn",
    main=DockerAdapter(image_name="ghcr.io/timeeval/knn", tag="0.3.1", skip_pull=True),
    preprocess=preprocess,
    data_as_file=True,
    training_type=TrainingType.UNSUPERVISED,
    input_dimensionality=InputDimensionality.MULTIVARIATE
    ),
    Algorithm(
    name="pfkde",
    main=DockerAdapter(image_name="pfkde", tag="latest", skip_pull=True),
    data_as_file=True,
    training_type=TrainingType.UNSUPERVISED,
    input_dimensionality=InputDimensionality.MULTIVARIATE,
    ),
]

repetitions = 1

#rcs = ResourceConstraints(task_memory_limit = 16 * GB, task_cpu_limit = 1.0,)

timeeval = TimeEval(dm, 
                    selected_datasets, 
                    algorithms, 
                    repetitions=repetitions, 
                    #resource_constraints=rcs, 
                    metrics=[
                        DefaultMetrics.ROC_AUC, 
                        F1Score(thresholding_strategy=PercentileThresholding(percentile= 100 * (1 - meta["contamination"]))), 
                        Precision(thresholding_strategy=PercentileThresholding(percentile= 100 * (1 - meta["contamination"]))), 
                        Recall(thresholding_strategy=PercentileThresholding(percentile= 100 * (1 - meta["contamination"])))
                        ]
                    )

timeeval.run()
results = timeeval.get_results()
print(results)


Running PREPARE phase
Running EVALUATION phase


Evaluating: 100%|██████████| 5/5 [1:45:45<00:00, 1269.16s/it]


Running FINALIZE phase
FINALIZE phase done.
          Stored results at C:\Users\Fi\Desktop\Code Folder\pfkde2\results\2026_06_23_08_55_28\results.csv.
          Overall runtime of this TimeEval run: 6346.085389137268 seconds
        
                                ROC_AUC_mean  \
algorithm collection   dataset                 
hbos      multivariate Bugsat       0.752617   
iforest   multivariate Bugsat       0.768189   
knn       multivariate Bugsat       0.542350   
pcc       multivariate Bugsat       0.589873   
pfkde     multivariate Bugsat       0.926626   

                                F1Score_PercentileThresholding(percentile=98.3277853925984)_mean  \
algorithm collection   dataset                                                                     
hbos      multivariate Bugsat                                            0.158594                  
iforest   multivariate Bugsat                                            0.251023                  
knn       multivariate Bugsa